<a href="https://colab.research.google.com/github/mohamedalaaaz/testpytroch/blob/main/Broker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

class TradingModel(nn.Module):
    def __init__(self, input_size):
        super(TradingModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output probability (up/down)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from model import TradingModel
import torch.nn as nn
import torch.optim as optim

def load_data(path):
    df = pd.read_csv(path)
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)  # Up = 1, Down = 0
    df.dropna(inplace=True)
    features = df[['Open', 'High', 'Low', 'Close', 'Volume']].values
    labels = df['Target'].values

    scaler = StandardScaler()
    features = scaler.fit_transform(features)

    X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2)

    return (
        DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                 torch.tensor(y_train, dtype=torch.float32)), batch_size=32, shuffle=True),
        DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                 torch.tensor(y_test, dtype=torch.float32)), batch_size=32),
        features.shape[1],  # input_size
        scaler
    )

def train_model():
    train_loader, test_loader, input_size, _ = load_data("data/historical_data.csv")
    model = TradingModel(input_size)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch).squeeze()
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")
    return model


In [ ]:
def simulate_trading(model, data_loader, capital=10000, threshold=0.6):
    model.eval()
    balance = capital
    shares = 0
    for X_batch, y_batch in data_loader:
        with torch.no_grad():
            probs = model(X_batch).squeeze().numpy()
        for i, prob in enumerate(probs):
            if prob > threshold and balance > 0:
                shares = balance / X_batch[i][3].item()  # Buy at current Close price
                balance = 0
            elif prob < 1 - threshold and shares > 0:
                balance = shares * X_batch[i][3].item()
                shares = 0
    final_value = balance + shares * X_batch[-1][3].item()
    print(f"Final Portfolio Value: ${final_value:.2f}")
